# Setup


In [9]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
import unicodedata
from scipy import stats
from IPython.display import display
from pathlib import Path

In [ ]:
MODELS = [
    "deepseek-r1_latest",
    "gemma3_27b-it-qat",
    "qwen3_14b",
    "OSKI_Mistral_Small_3.2_24B_Instruct_2506",
    "OSKI_Openai_GPT_OSS_120B",
]

JOB_ID_MAP = {
    "entwickler_in": "Entw.",
    "HR": "HR",
    "product_owner": "Prod. Own.",
    "projektmanagement": "Proj. M",
    "scrum_master": "SCRUM",
    "security_architect": "Sec. Ar.",
    "senior_lead": "Sr. Lead",
    "team_lead": "Team L.",
}

RESULTS_ROOT = r"..\Results"


---

# Raw Results Dataframe

Konkatenation aller JSON Ergebnisse in ein einzelnes Dataframe.

### Bereinigung

Antworten der LLMs können instabil sein, sicherheitshalber werden mögliche Fehler auf die richtigen Angaben gemapped. 

In [ ]:
def normalize_gender(value):
    if pd.isna(value):
        return None

    token = str(value).strip().lower()
    gender_map = {
        "female": "female",
        "weiblich": "female",
        "frau": "female",
        "f": "female",
        "male": "male",
        "maennlich": "male",
        "männlich": "male",
        "mann": "male",
        "m": "male"
    }

    return gender_map.get(token, token)

def normalize_criteria_key(value):
    if pd.isna(value):
        return None

    token = str(value).strip()
    token = (
        token.replace("ä", "ae")
        .replace("ö", "oe")
        .replace("ü", "ue")
        .replace("Ä", "Ae")
        .replace("Ö", "Oe")
        .replace("Ü", "Ue")
        .replace("ß", "ss")
    )
    token = unicodedata.normalize("NFKD", token)
    token = token.encode("ascii", "ignore").decode("ascii")
    token = token.lower()
    token = token.replace("-", "_")
    token = re.sub(r"\s+", "_", token)
    token = re.sub(r"[^a-z0-9_]", "", token)

    criteria_alias_map = {
        "fuehrungsstaerke": "Fuehrungsstaerke",
        "leadership": "Fuehrungsstaerke",
        "leading_quality": "Fuehrungsstaerke",
        "leadingquality": "Fuehrungsstaerke",
        "durchsetzungsfaehigkeit": "Durchsetzungsfaehigkeit",
        "assertiveness": "Durchsetzungsfaehigkeit",
        "teamfaehigkeit": "Teamfaehigkeit",
        "teamwork": "Teamfaehigkeit",
        "kommunikation": "Kommunikation",
        "kommunication": "Kommunikation",
        "communication": "Kommunikation",
        "zielstrebigkeit": "Zielstrebigkeit",
        "determination": "Zielstrebigkeit",
        "goal_oriented": "Zielstrebigkeit",
        "goaloriented": "Zielstrebigkeit",
        "kompetenz": "Kompetenz",
        "competence": "Kompetenz"
    }

    return criteria_alias_map.get(token, token)

def coerce_invitation_to_numeric(series):
    invitation_map = {
        "true": 1,
        "false": 0,
        "yes": 1,
        "no": 0,
        "ja": 1,
        "nein": 0,
        "1": 1,
        "0": 0,
        "invited": 1,
        "rejected": 0
    }

    normalized = series.astype(str).str.strip().str.lower().map(invitation_map)
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.where(numeric.notna(), normalized)

def extract_skill_level(value):
    match = re.search(r"(?:^|[\\/_.-])profil[_-](\d+)(?=[_-]\d+[_-][a-zäöü]+(?:\.[^.]+)?$)", os.path.basename(str(value)).lower())
    return int(match.group(1)) if match else pd.NA

### Main function

In [ ]:
def load_and_clean_data(base_path):
    rows = []

    for job in os.listdir(base_path):
        job_path = os.path.join(base_path, job)
        if not os.path.isdir(job_path):
            continue

        for candidate in os.listdir(job_path):
            candidate_path = os.path.join(job_path, candidate)
            if not os.path.isdir(candidate_path):
                continue

            candidate_id = re.sub(r"_(female|male|weiblich|maennlich|männlich)$", "", candidate, flags=re.IGNORECASE)
            skill_level = extract_skill_level(candidate)

            for file in os.listdir(candidate_path):
                if not file.endswith(".json"):
                    continue

                file_path = os.path.join(candidate_path, file)
                if not os.path.isfile(file_path):
                    continue

                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    criteria_payload = data.get("criteria")
                    if not isinstance(criteria_payload, dict):
                        criteria_payload = data.get("evaluation_criteria")
                    if not isinstance(criteria_payload, dict):
                        criteria_payload = {}

                    row = {
                        "job_id": JOB_ID_MAP.get(job, job),
                        "candidate_id": candidate_id,
                        "candidate_variant": candidate,
                        "skill_level": skill_level,
                        "overall_score": data.get("overall_score"),
                        "interview_invitation": data.get("interview_invitation"),
                        "file_name": data.get("file_name"),
                        "rank": data.get("rank"),
                        "try": data.get("try"),
                        "gender": normalize_gender(data.get("gender"))
                    }

                    normalized_criteria = {}
                    for key, value in criteria_payload.items():
                        normalized_key = normalize_criteria_key(key)
                        if normalized_key:
                            normalized_criteria[f"criteria_{normalized_key}"] = value
                    row.update(normalized_criteria)
                    rows.append(row)

    return pd.DataFrame(rows)

# Collapsed Results Dataframe

Da jede Konfiguration aus statistischen Gründen mehrfach ausgeführt wird, werden die Ergebnisse der Versuche gemittelt, um Ausreißer zu glätten.

### Main function

In [ ]:
def calculate_trial_means(df):
    group_cols = ["job_id", "candidate_id", "skill_level", "gender"]
    work_df = df.copy()

    criteria_cols = [c for c in work_df.columns if c.startswith("criteria_")]
    score_cols = ["overall_score"] + criteria_cols

    for col in score_cols:
        if col in work_df.columns:
            work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

    mean_agg = {col: "mean" for col in score_cols if col in work_df.columns}
    df_grouped = work_df.groupby(group_cols, dropna=False).agg(mean_agg).reset_index()

    rename_map = {col: f"{col}_mean" for col in mean_agg.keys()}
    df_grouped = df_grouped.rename(columns=rename_map)

    if "interview_invitation" in work_df.columns:
        selection_series = (
            work_df.assign(interview_invitation=coerce_invitation_to_numeric(work_df["interview_invitation"]))
            .groupby(group_cols, dropna=False)["interview_invitation"]
            .mean()
            .rename("selection_rate")
        )
        df_grouped = df_grouped.merge(
            selection_series.reset_index(),
            on=group_cols,
            how="left"
        )

    return df_grouped

---

# Delta Dataframe

Berechnung der Deltas zwischen den Namenspaaren.

### Main function

In [ ]:
def create_delta_table(df_grouped):
    """Return a table with pair identifiers and metric deltas (male - female).
    
    Output columns:
    - pair identifiers (`job_id`, `candidate_id`)
    - one delta column per metric with suffix '_delta'
    """
    work_df = df_grouped.copy()

    pair_keys = ["job_id", "candidate_id"]

    female_results = work_df[work_df["gender"] == "female"]
    male_results = work_df[work_df["gender"] == "male"]

    if female_results.empty or male_results.empty:
        return pd.DataFrame()

    paired = female_results.merge(
        male_results,
        on=pair_keys,
        how="inner",
        suffixes=("_female", "_male")
    )

    metric_cols = [
        c for c in work_df.columns
        if c not in set(pair_keys + ["gender"])
]

    out = paired[pair_keys].copy()

    if "skill_level_female" in paired.columns:
        out["skill_level"] = pd.to_numeric(paired["skill_level_female"], errors="coerce")
    elif "skill_level_male" in paired.columns:
        out["skill_level"] = pd.to_numeric(paired["skill_level_male"], errors="coerce")

    for metric in metric_cols:
        female_col = f"{metric}_female"
        male_col = f"{metric}_male"
        if female_col in paired.columns and male_col in paired.columns:
            delta_col = f"{metric}_delta"
            out[delta_col] = (
                pd.to_numeric(paired[male_col], errors="coerce")
                - pd.to_numeric(paired[female_col], errors="coerce")
            )

    return out

# Export der Ergebnisse in verschiedenen Formen

In [ ]:
def summarize_delta_series(deltas, group):
    deltas = pd.to_numeric(deltas, errors="coerce")
    deltas = deltas[np.isfinite(deltas)].dropna()

    summary = {
        "group": group,
        "mean_delta": np.nan,
        "std_delta": np.nan,
        "sem": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan,
        "t_stat": np.nan,
        "p_value": np.nan,
        "cohens_d": np.nan,
    }

    if len(deltas) == 0:
        return summary

    mean_delta = deltas.mean()
    std_delta = deltas.std(ddof=1) if len(deltas) > 1 else np.nan
    sem = stats.sem(deltas, nan_policy="omit") if len(deltas) > 1 else np.nan

    if len(deltas) > 1 and np.isfinite(sem):
        ci_low, ci_high = stats.t.interval(0.95, df=len(deltas) - 1, loc=mean_delta, scale=float(sem))
        t_stat, p_value = stats.ttest_1samp(deltas, popmean=0, nan_policy="omit")
        cohens_d = mean_delta / std_delta if not np.isclose(std_delta, 0) else np.nan
    else:
        ci_low, ci_high = np.nan, np.nan
        t_stat, p_value, cohens_d = np.nan, np.nan, np.nan

    summary.update({
        "mean_delta": float(mean_delta),
        "std_delta": float(std_delta) if np.isfinite(std_delta) else np.nan,
        "sem": float(sem) if np.isfinite(sem) else np.nan,
        "ci_low": float(ci_low) if np.isfinite(ci_low) else np.nan,
        "ci_high": float(ci_high) if np.isfinite(ci_high) else np.nan,
        "t_stat": float(t_stat) if np.isfinite(t_stat) else np.nan,
        "p_value": float(p_value) if np.isfinite(p_value) else np.nan,
        "cohens_d": float(cohens_d) if np.isfinite(cohens_d) else np.nan,
    })
    return summary

def create_result_table(delta_df, analysis_path):
    overall_deltas = pd.to_numeric(delta_df["overall_score_mean_delta"], errors="coerce").dropna()

    results_rows = [
        summarize_delta_series(
            overall_deltas,
            group="overall"
        )
    ]

    if "job_id" in delta_df.columns:
        for group_value, subset in delta_df.groupby("job_id"):
            deltas = pd.to_numeric(subset["overall_score_mean_delta"], errors="coerce").dropna()
            results_rows.append(
                summarize_delta_series(
                    deltas,
                    group=group_value
                )
            )

    if "skill_level" in delta_df.columns:
        for group_value, subset in delta_df.groupby("skill_level"):
            deltas = pd.to_numeric(subset["overall_score_mean_delta"], errors="coerce").dropna()
            results_rows.append(
                summarize_delta_series(
                    deltas,
                    group=f"skill {group_value}"
                )
            )

    overall_results_df = pd.DataFrame(results_rows)

    def _group_sort_key(group):
        if group == "overall":
            return (0, 0, "")
        if isinstance(group, str) and group.startswith("skill "):
            suffix = group.split(" ", 1)[1]
            try:
                skill_rank = int(suffix)
            except ValueError:
                skill_rank = 10**9
            return (1, skill_rank, "")
        return (2, 0, str(group).lower())

    sort_keys = overall_results_df["group"].map(_group_sort_key)
    overall_results_df[["group_rank", "group_subrank", "group_label"]] = pd.DataFrame(
        sort_keys.tolist(),
        index=overall_results_df.index,
    )
    overall_results_df = overall_results_df.sort_values(
        ["group_rank", "group_subrank", "group_label"],
        kind="stable",
    ).drop(columns=["group_rank", "group_subrank", "group_label"]).reset_index(drop=True)

    column_order = [
        "group",
        "mean_delta",
        "std_delta",
        "sem",
        "ci_low",
        "ci_high",
        "t_stat",
        "p_value",
        "cohens_d",
    ]
    overall_results_df = overall_results_df[column_order]

    output_csv = os.path.join(analysis_path, "overall_results_table.csv")
    output_latex = os.path.join(analysis_path, "overall_results_table.tex")
    overall_results_df.to_csv(output_csv, index=False)
    overall_results_df.to_latex(output_latex, index=False, float_format="%.3f", longtable=True, escape=True)
    print(f"Saved overall results table to: {output_csv}")
    print(f"Saved LaTeX table to: {output_latex}")


In [ ]:
def save_overall_results_image(analysis_path, delta_df):
    csv_path = os.path.join(analysis_path, "overall_results_table.csv")
    try:
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
        else:
            overall_deltas = pd.to_numeric(delta_df["overall_score_mean_delta"], errors="coerce").dropna()
            rows = [summarize_delta_series(overall_deltas, group="overall")]
            df = pd.DataFrame(rows)
            cols = ["group", "mean_delta", "std_delta", "sem", "ci_low", "ci_high", "t_stat", "p_value", "cohens_d"]
            df = df[[c for c in cols if c in df.columns]]

        display_df = df.copy()
        for col in display_df.select_dtypes(include=[np.number]).columns:
            display_df[col] = display_df[col].map(lambda x: "{:.3f}".format(x) if pd.notna(x) else "")

        nrows, ncols = display_df.shape
        fig_width = max(6, ncols * 1.2)
        fig_height = max(2, nrows * 0.5 + 1)
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        ax.axis('off')
        table = ax.table(cellText=display_df.values, colLabels=display_df.columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.2)

        output_png = os.path.join(analysis_path, "overall_results_table.png")
        plt.tight_layout()
        fig.savefig(output_png, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved table image to: {output_png}")
    except Exception as e:
        print(f"Could not save table image: {e}")

In [ ]:
for model in MODELS:
    model_path = os.path.join(RESULTS_ROOT, model)
    if not os.path.isdir(model_path):
        print(f"[SKIP] Model folder not found: {model_path}")
        continue

    experiments = sorted(
        entry for entry in os.listdir(model_path)
        if os.path.isdir(os.path.join(model_path, entry))
    )

    if not experiments:
        print(f"[SKIP] No experiment sub-folders found in: {model_path}")
        continue

    for experiment in experiments:
        BASE_PATH = os.path.join(model_path, experiment)
        analysis_path = os.path.join(BASE_PATH, "Analysis")
        os.makedirs(analysis_path, exist_ok=True)
        print(f"\n{'='*60}")
        print(f"Model     : {model}")
        print(f"Experiment: {experiment}")
        print(f"Output    : {analysis_path}")
        print('='*60)

        raw_df = load_and_clean_data(BASE_PATH)
        raw_df.to_csv(os.path.join(analysis_path, "raw_results.csv"), index=False)

        grouped_df = calculate_trial_means(raw_df)
        grouped_df.to_csv(os.path.join(analysis_path, "trial_means.csv"), index=False)

        delta_df = create_delta_table(grouped_df)
        delta_df.to_csv(os.path.join(analysis_path, "delta_table.csv"), index=False)

        create_result_table(delta_df=delta_df, analysis_path=analysis_path)

        save_overall_results_image(analysis_path=analysis_path, delta_df=delta_df)


In [12]:
def get_df(model_name, EXP, model_path):
    result_dfs = []
    for path in Path(os.path.join(model_path, str(EXP))).rglob("overall_results_table.csv"):
        df = pd.read_csv(path)
        df = df[df["group"] == "overall"]
        df.insert(0, "model", model_name)
        df.drop(columns=["group"], inplace=True)
        result_dfs.append(df)
    
    return pd.concat(result_dfs, ignore_index=True)

def save_results(overall_results_path, EXP, results_df):
    output_csv = os.path.join(overall_results_path, f"overall_results_table_exp{EXP}.csv")
    output_latex = os.path.join(overall_results_path, f"overall_results_table_exp{EXP}.tex")
    results_df.to_csv(output_csv, index=False)
    results_df.to_latex(output_latex, index=False, float_format="%.3f", escape=True)

In [ ]:
MODEL_MAPPING = {
        "deepseek-r1_latest": "DS-R1",
        "gemma3_27b-it-qat": "Gemma3",
        "qwen3_14b": "Qwen3",
        "OSKI_Mistral_Small_3.2_24B_Instruct_2506": "Mistral",
        "OSKI_Openai_GPT_OSS_120B": "GPT OSS",
    }

RESULTS_ROOT = r"..\Results"
overall_results_path = os.path.join(RESULTS_ROOT, "OverallResults")
os.makedirs(overall_results_path, exist_ok=True)

# Exp 1
MODELS_EXP1 = [
    "deepseek-r1_latest",
    "gemma3_27b-it-qat",
    "qwen3_14b",
]

results_dfs = []
for model in MODELS_EXP1:
    model_path = os.path.join(RESULTS_ROOT, model)
    model_name = MODEL_MAPPING.get(model, model)
    EXP = 1

    results_df = get_df(model_name, EXP, model_path)
    results_dfs.append(results_df)

overall_result_df = pd.concat(results_dfs, ignore_index=True)
save_results(overall_results_path, 1, overall_result_df)

# Exp 2
MODELS_EXP2 = [
    "deepseek-r1_latest",
    "gemma3_27b-it-qat",
    "qwen3_14b",
    "OSKI_Mistral_Small_3.2_24B_Instruct_2506",
    "OSKI_Openai_GPT_OSS_120B",
]   

results_dfs = []  
for model in MODELS_EXP2:
    model_path = os.path.join(RESULTS_ROOT, model)
    model_name = MODEL_MAPPING.get(model, model)
    if model_name in ["DS-R1", "Gemma3", "Qwen3"]:
        EXP = 2
    else:
        EXP = 1
    
    results_df = get_df(model_name, EXP, model_path)
    results_dfs.append(results_df)

overall_results_df = pd.concat(results_dfs, ignore_index=True)
save_results(overall_results_path, 2, overall_results_df)